# Workshop 1: LoRA Fine-Tuning with Qwen

Fine-tune a small language model using **LoRA** (Low-Rank Adaptation) on the Alpaca instruction dataset.  
We train only ~0.3% of parameters while the base model stays frozen.

**What you'll do:**
1. Load and format an instruction dataset  
2. Tokenize prompts  
3. Load a quantized base model (QLoRA)  
4. Attach a LoRA adapter  
5. Train with early stopping  
6. Plot train / validation / test loss curves  
7. Save and reload the adapter for inference

## 0. Install dependencies

Run once, then restart the kernel.

In [ ]:
# !pip install torch transformers datasets accelerate peft bitsandbytes safetensors tokenizers matplotlib

## 1. Imports

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    default_data_collator,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from datasets import load_dataset, DatasetDict

## 2. Choose the base model

Change this one string to swap models — nothing else needs to change.

| Model | Size | Notes |
|---|---|---|
| `Qwen/Qwen2.5-0.5B-Instruct` | 0.5 B | Default — runs on CPU or any GPU |
| `Qwen/Qwen3.5-27B` | 27 B | Needs A10+ |
| `google/gemma-3-27b-it` | 27 B | Google Gemma instruction model |
| `mistralai/Mistral-7B-Instruct-v0.3` | 7 B | Original workshop default |

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

## 3. Prompt formatting

`build_prompt` converts an Alpaca example into the `### Instruction / Input / Response` format the model expects.

In [ ]:
def build_prompt(example):
    instruction = example["instruction"].strip()
    user_input  = example["input"].strip()
    output      = example["output"].strip()

    if user_input:
        return (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{user_input}\n\n"
            f"### Response:\n{output}"
        )
    return f"### Instruction:\n{instruction}\n\n### Response:\n{output}"


# Quick preview
sample = {
    "instruction": "Summarize the paragraph.",
    "input": "The Eiffel Tower is in Paris.",
    "output": "The Eiffel Tower is a Paris landmark."
}
print(build_prompt(sample))

## 4. Load dataset and create train / validation / test splits

We use 600 examples from `yahma/alpaca-cleaned` — enough to see meaningful learning in a workshop.

**Option B** (commented below): load your own local JSON or CSV file instead.

In [ ]:
print("Loading dataset...")

# --- Option A: public HuggingFace dataset ---
raw_dataset   = load_dataset("yahma/alpaca-cleaned")
small_dataset = raw_dataset["train"].shuffle(seed=42).select(range(600))

# --- Option B: local JSON or CSV ---
# raw_dataset   = load_dataset("json", data_files="path/to/your_data.json")
# # or: raw_dataset = load_dataset("csv", data_files="path/to/your_data.csv")
# # Your file must have 'instruction', 'input', and 'output' columns.
# small_dataset = raw_dataset["train"]   # already a flat split

# --- Split: 80% train / 10% val / 10% test ---
train_test = small_dataset.train_test_split(test_size=0.20, seed=42)
valid_test = train_test["test"].train_test_split(test_size=0.50, seed=42)

data = DatasetDict({
    "train":      train_test["train"],
    "validation": valid_test["train"],
    "test":       valid_test["test"],
})

print(data)
# Expected: ~480 train / 60 validation / 60 test

## 5. Tokenizer and tokenization

- `max_length=512` — truncate/pad every sample to 512 tokens (halves memory vs 1024)  
- `labels = input_ids` — for causal LM, the target is the same sequence shifted by one (handled internally by `Trainer`)

In [ ]:
print("Preparing tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

# Some models don't define a pad token; reuse EOS safely
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def tokenize_function(example):
    prompt    = build_prompt(example)
    tokenized = tokenizer(
        prompt,
        truncation=True,
        max_length=512,
        padding="max_length",
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized


tokenized = data.map(tokenize_function, batched=False)
print("Tokenization complete.", tokenized)

## 6. Load the base model with 4-bit quantization (QLoRA)

Weights are stored as 4-bit integers (~8× smaller in memory).  
Computation is still done in `bfloat16` for numerical stability.  
The base model weights are **frozen** — only the LoRA adapter will be trained.

In [ ]:
print("Loading base model with 4-bit quantization...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,    # quantize the quantization constants too
    bnb_4bit_quant_type="nf4",         # Normal Float 4: optimal for neural network weights
    bnb_4bit_compute_dtype=torch.bfloat16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",                 # automatically place layers on available GPU/CPU
)

# Prepares the quantized model for LoRA training:
# casts layer norms to float32 and enables gradient checkpointing
base_model = prepare_model_for_kbit_training(base_model)
print("Base model loaded.")

## 7. Attach the LoRA adapter

**`task_type`** tells PEFT what problem we are solving:

| task_type | Use for | Example models |
|---|---|---|
| `CAUSAL_LM` | Decoder-only text generation | GPT, LLaMA, Qwen, Mistral |
| `SEQ_2_SEQ_LM` | Encoder-decoder generation | T5, BART, FLAN-T5 |
| `SEQ_CLASSIFICATION` | Sentence/document classification | BERT, RoBERTa |
| `TOKEN_CLS` | Token-level labeling (e.g. NER) | BERT, DistilBERT |
| `QUESTION_ANSWERING` | Span extraction QA | BERT, ALBERT |

**`inference_mode=False`** builds the gradient graph for training.  
Set to `True` when loading the adapter for inference to save memory.

**`r=16`** — rank of each adapter matrix pair.  
**`lora_alpha=32`** — scales adapter output by `alpha/r = 2.0`.  
**`target_modules`** — attach adapters to these four attention projection layers.

In [ ]:
# The task_type tells PEFT what kind of problem we are solving.
# Common values:
#   CAUSAL_LM         - decoder-only text generation (GPT, LLaMA, Mistral, Qwen)
#   SEQ_2_SEQ_LM      - encoder-decoder generation (T5, BART)
#   SEQ_CLASSIFICATION - sentence/document classification
#   TOKEN_CLS         - token-level classification (NER)
#   QUESTION_ANSWERING - span extraction QA
lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    inference_mode=False,   # True when loading for inference (saves memory)
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
# Expected: trainable params ~1.3M || all params ~500M || trainable% ~0.26%

## 8. MultiEvalTrainer — track test loss as a full curve

The standard `Trainer` only evaluates one dataset at a time.  
`MultiEvalTrainer` overrides `evaluate()` to also run the test set at every eval step,
so we get a full **test loss curve** (not just a single endpoint).

The `if eval_dataset is None` guard ensures test eval only runs during the training loop,
not on any manual `.evaluate()` calls.

In [ ]:
class MultiEvalTrainer(Trainer):
    """Trainer that also evaluates the test set at every eval step."""

    def __init__(self, test_dataset=None, **kwargs):
        super().__init__(**kwargs)
        self._test_dataset = test_dataset

    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval"):
        metrics = super().evaluate(
            eval_dataset=eval_dataset,
            ignore_keys=ignore_keys,
            metric_key_prefix=metric_key_prefix,
        )
        # Only run test eval during training-loop evals, not on explicit external calls
        if eval_dataset is None and self._test_dataset is not None:
            test_metrics = super().evaluate(
                eval_dataset=self._test_dataset,
                ignore_keys=ignore_keys,
                metric_key_prefix="test",
            )
            self.log({"test_loss": test_metrics["test_loss"]})
        return metrics

## 9. Training arguments and trainer setup

**Gradient accumulation:** `batch_size=1` × `accumulation_steps=16` → effective batch of 16 without holding all 16 in VRAM at once.

**Early stopping:** if validation loss doesn't improve for 3 consecutive eval checkpoints (every 5 steps = patience of 15 steps), training stops and the best checkpoint is restored.

In [ ]:
os.makedirs("workshop_outputs/lora-alpaca", exist_ok=True)

training_args = TrainingArguments(
    output_dir="workshop_outputs/lora-alpaca",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,    # effective batch size = 16
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=5,                      # evaluate every 5 steps
    save_steps=5,
    save_total_limit=2,
    load_best_model_at_end=True,       # restore best checkpoint when training ends
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    optim="paged_adamw_32bit",
    report_to="none",
)

trainer = MultiEvalTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    test_dataset=tokenized["test"],
    processing_class=tokenizer,
    data_collator=default_data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("Trainer ready.")

## 10. Train!

In [ ]:
trainer.train()

## 11. Plot train / validation / test loss curves

**What to look for:**
- **Train loss (blue):** Should decrease steadily.  
- **Validation loss (orange):** Tracks train loss early, then plateaus. Rising sharply = overfitting.  
- **Test loss (red):** Should stay close to validation loss. A large gap = non-representative split.

In [ ]:
train_steps, train_losses = [], []
eval_steps,  eval_losses  = [], []
test_steps,  test_losses  = [], []

for entry in trainer.state.log_history:
    if "loss" in entry and "eval_loss" not in entry:
        train_steps.append(entry["step"])
        train_losses.append(entry["loss"])
    if "eval_loss" in entry:
        eval_steps.append(entry["step"])
        eval_losses.append(entry["eval_loss"])
    if "test_loss" in entry:
        test_steps.append(entry["step"])
        test_losses.append(entry["test_loss"])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_steps, train_losses, label="Train loss",      color="steelblue",  linewidth=2)
ax.plot(eval_steps,  eval_losses,  label="Validation loss", color="darkorange", linewidth=2, marker="o", markersize=5)
ax.plot(test_steps,  test_losses,  label="Test loss",       color="crimson",    linewidth=2, marker="s", markersize=5)
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("LoRA Fine-Tuning — Train / Validation / Test Loss")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()

plot_path = "workshop_outputs/lora-alpaca/loss_curves.png"
fig.savefig(plot_path, dpi=150)
plt.show()
print(f"Loss plot saved to {plot_path}")

## 12. Save the LoRA adapter

Only the adapter weights are saved (~5 MB), not the full base model (~500 MB).  
Anyone with the base model can use your adapter file.

In [ ]:
trainer.save_model("lora_alpaca_adapter")
tokenizer.save_pretrained("lora_alpaca_adapter")
print("Saved LoRA adapter to lora_alpaca_adapter/")
# Files created:
#   adapter_config.json        — records r, alpha, target_modules, base model name
#   adapter_model.safetensors  — the trained adapter weights
#   tokenizer files

## 13. Reload the adapter and run inference

In [ ]:
base_for_inference = AutoModelForCausalLM.from_pretrained(
    model_name, device_map="auto"
)
inference_model     = PeftModel.from_pretrained(base_for_inference, "lora_alpaca_adapter")
inference_tokenizer = AutoTokenizer.from_pretrained("lora_alpaca_adapter")

# Use an instruction from the test set
test_example = data["test"][0]
prompt = build_prompt({**test_example, "output": ""}).rstrip()

inputs  = inference_tokenizer(prompt, return_tensors="pt").to(inference_model.device)
outputs = inference_model.generate(**inputs, max_new_tokens=200)
print(inference_tokenizer.decode(outputs[0], skip_special_tokens=True))

## 14. Extension ideas

- Swap `model_name` to `mistralai/Mistral-7B-Instruct-v0.3` and compare results  
- Use Option B to load your own instruction dataset  
- Increase `r` and `lora_alpha` — observe the effect on loss and adapter file size  
- Merge the adapter permanently: `model.merge_and_unload()` collapses LoRA into the base weights  
- Try `task_type="SEQ_CLASSIFICATION"` on a BERT model for text classification